# PXRD_Viewer_SSA
### PXRD Interactice Viewer for previously plotted 1D data 
 - use 07_PXRD_Pattern_Stacking_SSA.ipynb first to get plots that can then be loaded in here. 

In [ ]:
# imports 

import pandas as pd 
import plotly.graph_objects as go 
import os 

In [ ]:
#Load data made in 07_PXRD_Pattern_stacking_SSA 

data_folder = "E:/CBZ" #"D:/route/to/stack/pxrd/plot/"
excel_path = os.path.join(data_folder, "07_Pir_ref_comp_stacked_patterns.xlsx") #link to excel file within this folder

df = pd.read_excel(excel_path)
print(df.columns)


In [ ]:
# Identify reference patterns and optionally set a manual plotting order (bottom -> top)
from difflib import get_close_matches

cols = df.columns[1:]  # skip the '2theta' column
ref_cols = [c for c in cols if "ref" in c.lower() or "dolbir" in c.lower()]
nonref_cols = [c for c in cols if c not in ref_cols]

# OPTIONAL: define exact column order from bottom to top.
# Leave empty ([]) to use the automatic ordering below.
manual_bottom_to_top = [
    "DLM Alpha ref",
    "DLM Beta ref",
    "DL-Methionine 0.079VF (EtOH)",
    "DL-Methionine 0.05VF (EtOH)",
    ]

def _norm_name(name):
    # Normalize minor differences such as case and repeated spaces.
    return " ".join(str(name).split()).lower()

if manual_bottom_to_top:
    col_lookup = {_norm_name(c): c for c in cols}
    resolved_manual = []
    unmatched = []

    for requested in manual_bottom_to_top:
        key = _norm_name(requested)
        matched = col_lookup.get(key)
        if matched is not None:
            resolved_manual.append(matched)
        else:
            suggestions = get_close_matches(requested, list(cols), n=3, cutoff=0.6)
            unmatched.append((requested, suggestions))

    if unmatched:
        print("Warning: Some manual names were not found and were skipped:")
        for bad, suggestions in unmatched:
            if suggestions:
                print(f"  - {bad} (closest: {suggestions})")
            else:
                print(f"  - {bad} (no close match)")

    if resolved_manual:
        # Keep resolved manual order first, then append any remaining columns.
        remaining_cols = [c for c in cols if c not in resolved_manual]
        ordered_cols = resolved_manual + remaining_cols
    else:
        print("Warning: No valid manual names found. Falling back to automatic ordering.")
        ordered_cols = ref_cols + nonref_cols
else:
    # Automatic ordering: references at bottom, then non-reference patterns.
    ordered_cols = ref_cols + nonref_cols

In [ ]:
#Interactive GUI Stack Plot 
fig = go.Figure()
offset = 0
offset_step = df.iloc[:, 1:].max().max() * 1.1  # <<---vertical offset - change accordingly 

for col in ordered_cols:
    fig.add_trace(go.Scatter(
        x=df["2theta"],
        y=df[col] + offset,
        mode='lines',
        name=col,
        line=dict(width=1.5),
        hovertemplate=f"<b>{col}</b><br>2θ: %{{x:.2f}}°<br>Intensity: %{{y:.0f}}<extra></extra>"
    ))
    offset += offset_step

#Make the figure 
fig.update_layout(
    #title="PXRD Interactive Stack Plot",
    xaxis_title="2θ (degrees)",
    #yaxis_title="Intensity (offset)",
    template="plotly_white",
    hovermode="closest",
    legend=dict(
        x=1.,
        y=.95,
        xanchor="right",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",
        traceorder="reversed"  
    ),
    height=800,
)

fig.update_xaxes(automargin=True, tickmode="auto", nticks=15) #tick marks 


print ("Click on Legend to Select/ De-select Data Set")


fig.update_yaxes(showticklabels=False)#, title_text="Intensity (offset)")
fig.update_layout(
    font=dict(size=14),            # global font size
    xaxis_title_font=dict(size=18),
    yaxis_title_font=dict(size=18),
)


fig.show()


In [ ]:
#Export stacked PXRD data to Excel + Interactive HTML <<-- Data to send Karen 
# Define save paths
stacked_excel_path = os.path.join(data_folder, "polymorph_comparison_stacked_powder_patterns.xlsx")
html_path = os.path.join(data_folder, "polymorph_comparison_interactive_stack_plot.html")

#1. Save Excel file (2theta + each dataset as a column)
# --- Copy the dataframe so we can modify it for stacked view ---
df_stacked = df.copy()

# --- Define vertical offset ---
# You can adjust this factor depending on your data intensity range
max_intensity = df_stacked.iloc[:, 1:].max().max()
offset_step = max_intensity * 1.1  # same as used in interactive plot

# --- Apply offsets to each pattern column (skip 2theta) ---
for i, col in enumerate(df_stacked.columns[1:]):
    df_stacked[col] = df_stacked[col] + i * offset_step

# --- Save to Excel ---
df_stacked.to_excel(stacked_excel_path, index=False)

print(f"✅ Stacked Excel file saved and ready to view: {stacked_excel_path}")

#2. Save interactive Plotly viewer as HTML
fig.write_html(html_path, include_plotlyjs='cdn')


print(f"✅ Interactive HTML viewer saved to:\n{html_path}")
print("\nBoth files are in your data_folder and can be opened directly (Excel for data, browser for the interactive plot).")
